In [1]:
def extract_sla_details(row):
    """
    Simulates LLM-based SLA extraction.
    Returns SLA details in JSON format.
    """
    return {
        "monthly_emi": row.get("monthly_emi"),
        "interest_rate": row.get("interest_rate"),
        "tenure_months": row.get("tenure_months"),
        "risk_flag": row.get("risk_flag"),
        "issue_type": row.get("issue_type"),
        "recommended_action": row.get("recommended_action")
    }

In [4]:
import pandas as pd

contracts_df = pd.read_csv("..\data\sample_car_contracts_with_vin.csv")
contracts_df.head()

<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
C:\Users\DELL\AppData\Local\Temp\ipykernel_16240\1892078674.py:3: SyntaxWarning: invalid escape sequence '\d'
  contracts_df = pd.read_csv("..\data\sample_car_contracts_with_vin.csv")


,id,customer_name,contract_type,vehicle_type,monthly_emi,interest_rate,tenure_months,clause_summary,risk_flag,issue_type,recommended_action,vin
0,1,Rahul Mehta,Car Loan,Sedan,18500,9.5,48,Prepayment allowed only after 24 months with 5...,medium,High prepayment charges,Highlight prepayment penalty to user and sugge...,1HGCM82633A004352
1,2,Anita Rao,Car Lease,SUV,22000,0.0,36,Lessee must pay for all maintenance and insurance,low,Standard maintenance clause,"No action, just explain maintenance responsibi...",1HGCM82633A004352
2,3,James Wilson,Car Loan,Hatchback,14500,11.2,60,Late payment fee of 3% per month on outstandin...,high,Aggressive late fee,Flag clause and suggest user request cap on la...,1HGCM82633A004352
3,4,Meena Iyer,Car Lease,Sedan,21000,0.0,24,"Excess mileage charge of ₹12 per km over 15,00...",medium,High excess mileage rate,Warn user about extra mileage charges and reco...,1HGCM82633A004352
4,5,Arjun Patel,Car Loan,SUV,27500,10.8,72,Floating interest rate linked to lender's inte...,high,Unclear interest benchmark,Explain floating rate risk and suggest asking ...,1HGCM82633A004352


In [5]:
sla_records = []

for _, row in contracts_df.iterrows():
    sla_records.append({
        "contract_id": row["id"],
        "customer_name": row["customer_name"],
        "sla": extract_sla_details(row)
    })

sla_df = pd.DataFrame(sla_records)
sla_df.head()

,contract_id,customer_name,sla
0,1,Rahul Mehta,"{'monthly_emi': 18500, 'interest_rate': 9.5, '..."
1,2,Anita Rao,"{'monthly_emi': 22000, 'interest_rate': 0.0, '..."
2,3,James Wilson,"{'monthly_emi': 14500, 'interest_rate': 11.2, ..."
3,4,Meena Iyer,"{'monthly_emi': 21000, 'interest_rate': 0.0, '..."
4,5,Arjun Patel,"{'monthly_emi': 27500, 'interest_rate': 10.8, ..."


In [6]:
sla_df.to_csv("../data/Milestone2_sla_output.csv", index=False)

In [7]:
sla_df.sample(6)

,contract_id,customer_name,sla
5,6,Sana Khan,"{'monthly_emi': 16500, 'interest_rate': 0.0, '..."
2,3,James Wilson,"{'monthly_emi': 14500, 'interest_rate': 11.2, ..."
6,7,Vikram Shah,"{'monthly_emi': 19500, 'interest_rate': 8.9, '..."
8,9,David Roy,"{'monthly_emi': 30500, 'interest_rate': 12.5, ..."
10,11,Pratima Das,"{'monthly_emi': 13200, 'interest_rate': 9.9, '..."
0,1,Rahul Mehta,"{'monthly_emi': 18500, 'interest_rate': 9.5, '..."


In [8]:
import requests

def get_vehicle_details(vin):
    url = f"https://vpic.nhtsa.dot.gov/api/vehicles/DecodeVinValues/{vin}?format=json"
    response = requests.get(url)
    return response.json()["Results"][0]

In [9]:
def extract_vehicle_info(vehicle_raw):
    if not vehicle_raw:
        return {"make": None, "model": None, "year": None}

    return {
        "make": vehicle_raw.get("Make"),
        "model": vehicle_raw.get("Model"),
        "year": vehicle_raw.get("ModelYear")
    }

In [10]:
def enrich_contract_with_vehicle(row):
    vehicle_raw = get_vehicle_details(row["vin"])
    vehicle_info = extract_vehicle_info(vehicle_raw)

    return {
        "contract_id": row.get("contract_id") or row.get("id"),
        "sla": extract_sla_details(row),
        "vehicle": vehicle_info
    }

In [11]:
combined_records = []

for _, row in contracts_df.iterrows():
    combined_records.append(enrich_contract_with_vehicle(row))

combined_df = pd.DataFrame(combined_records)
combined_df.head()

,contract_id,sla,vehicle
0,1,"{'monthly_emi': 18500, 'interest_rate': 9.5, '...","{'make': 'HONDA', 'model': 'Accord', 'year': '..."
1,2,"{'monthly_emi': 22000, 'interest_rate': 0.0, '...","{'make': 'HONDA', 'model': 'Accord', 'year': '..."
2,3,"{'monthly_emi': 14500, 'interest_rate': 11.2, ...","{'make': 'HONDA', 'model': 'Accord', 'year': '..."
3,4,"{'monthly_emi': 21000, 'interest_rate': 0.0, '...","{'make': 'HONDA', 'model': 'Accord', 'year': '..."
4,5,"{'monthly_emi': 27500, 'interest_rate': 10.8, ...","{'make': 'HONDA', 'model': 'Accord', 'year': '..."


In [12]:
print(contracts_df.columns.tolist())

['id', 'customer_name', 'contract_type', 'vehicle_type', 'monthly_emi', 'interest_rate', 'tenure_months', 'clause_summary', 'risk_flag', 'issue_type', 'recommended_action', 'vin']


In [ ]:
combined_df.to_csv("../data/milestone2_evaluation_output.csv", index=False)